In [1]:
import json
from pathlib import Path
from collections import Counter
import tabulate
from tqdm import tqdm

In [3]:
subjs_details = {}

subjs_appr_100 = []
with open('../code/subjects/approach_100.json') as f:
    for name in json.load(f):
        subjs_appr_100.append(name)

subjs_eval_200 = []
with open('../code/subjects/eval_200.json') as f:
    for name in json.load(f):
        subjs_eval_200.append(name)

MSB_FLASH_FILES = Path('~/Multi-SWE-bench-flash').expanduser()

subjs_multi_flash = []
with (MSB_FLASH_FILES / 'multi_swe_bench_flash.jsonl').open() as f:
    for l in f.read().splitlines():
        d = json.loads(l)
        subjs_multi_flash.append(d['instance_id'])
        subjs_details[d['instance_id']] = d

In [4]:
def get_metrics(base_name, bug_id, extra_keys):
    p = Path('trajs') / base_name / f'{bug_id}.json'
    if not p.is_file():
        return None

    with p.open() as f:
        d = json.load(f)

    #if d['result']['gen'] in ["err-<class 'openai.PermissionDeniedError'>", "err-<class 'openai.BadRequestError'>", "err-<class 'ZeroDivisionError'>", "err-<class 'openai.NotFoundError'>"]:
    #    return None

    metrics = {k: d['metrics'][k] for k in [
        'tot_step',
        'cost_tokens', 'prompt_tokens', 'completion_tokens',
        *extra_keys,
    ]}

    if d['result']['gen']=="err-<class 'openai.APIStatusError'>":
        metrics['prompt_tokens'] += 200_000 # "prompt too long"

    return {
        'result_gen': d['result']['gen'],
        'result_val': d['result']['val'],
        'final_step': sum((1 if m['role']=='assistant' else 0) for m in d['messages']),
        **metrics,
    }

In [5]:
def draw_table(rows):
    headers = []
    for r in rows.values():
        for k in r.keys():
            if k not in headers:
                headers.append(k)

    table = []
    for k, r in rows.items():
        table.append([k, *[
            r.get(h, None) for h in headers
        ]])

    #return tabulate.tabulate(table, headers=headers, tablefmt='html', disable_numparse=True)
    print(tabulate.tabulate(table, headers=headers, tablefmt='tsv', disable_numparse=True))

In [6]:
ANALYSIS_SYS_PROMPT_TOKENS = 492

ANALYSIS_PRICES = {
    'llm_claude4sonnet': (.3, 3, 15),
    'llm_claude35haiku': (.08, .8, 4),
    'llm_gemini25pro': (.31, 1.25, 10),
    'llm_gemini25flash': (.075, .3, 2.5),
    'llm_gpt5mini': (.03, .25, 2),
    'llm_deepseekv3': (.07, .27, 1.1),
    'llm_qwen3': (.07, .27, 1.1),
}

M = 1e6
LINGUA_PRICE = .01

EXTRA_KEYS_LINGUA = [
    'analysis_count', 'erase_tot_count',
    'seen_tokens', 'erase_in_tokens', 'erase_out_tokens',
]
EXTRA_KEYS_OURS = EXTRA_KEYS_LINGUA + [
    'analysis_cost_tokens', 'analysis_prompt_tokens', 'analysis_completion_tokens',
]

def export(subjs, baselines, fix_agent_name='llm_claude4sonnet'):
    tot_cnt = Counter()
    pass_cnt = Counter()
    step_cnt = Counter()
    passed_step_cnt = Counter()
    keep_cnt = Counter()
    ctx_cnt = Counter()
    agent_in_cnt = Counter()
    agent_out_cnt = Counter()
    agent_cost_cnt = Counter()
    overhead_in_cnt = Counter()
    overhead_out_cnt = Counter()
    overhead_cost_cnt = Counter()

    AGENT_PRICE = ANALYSIS_PRICES[fix_agent_name]

    for bug_id in tqdm(subjs):
        for name, base_name, extra_keys in baselines:
            llm_key = 'llm_' + name
            if llm_key not in ANALYSIS_PRICES:
                #llm_key = 'llm_gemini25flash'
                llm_key = 'llm_gpt5mini'

            ret = get_metrics(base_name, bug_id, extra_keys)
            if ret:
                tot_cnt[name] += 1

                if ret['result_val']=='pass':
                    pass_cnt[name] += 1
                    passed_step_cnt[name] += ret['tot_step']

                step_cnt[name] += ret['tot_step']

                #if ret['result_gen'].startswith('err'):
                #    print('WARN: gen', name, bug_id, ret['result_gen'])
                #if ret['result_val'].startswith('err'):
                #    print('WARN: val', name, bug_id, ret['result_val'])

                agent_in_cnt[name] += ret['prompt_tokens']
                agent_out_cnt[name] += ret['completion_tokens']
                agent_cost_cnt[name] += ret['prompt_tokens'] * AGENT_PRICE[0] + ret['prompt_tokens'] * .02 * AGENT_PRICE[1] + ret['completion_tokens'] * AGENT_PRICE[2]

                if 'seen_tokens' in ret:
                    tot = ret['seen_tokens']
                    keep_cnt[name] += (ret['erase_out_tokens'] / ret['erase_in_tokens']) if ret['erase_in_tokens'] else 1
                    ctx_cnt[name] += ((tot - ret['erase_in_tokens'] + ret['erase_out_tokens']) / tot) if tot else 1

                if 'analysis_prompt_tokens' in ret:
                    ANALYSIS_PRICE = ANALYSIS_PRICES[llm_key]
                    adjustment = ret['analysis_count'] * ANALYSIS_SYS_PROMPT_TOKENS # system prompt can be cached
                    overhead_in_cnt[name] += ret['analysis_prompt_tokens'] - adjustment
                    overhead_out_cnt[name] += ret['analysis_completion_tokens']
                    overhead_cost_cnt[name] += ret['analysis_prompt_tokens'] * ANALYSIS_PRICE[1] - adjustment * (ANALYSIS_PRICE[1] - ANALYSIS_PRICE[0]) + ret['analysis_completion_tokens'] * ANALYSIS_PRICE[2]

                elif 'lingua' in name:
                    overhead_in_cnt[name] += ret['erase_in_tokens']
                    overhead_cost_cnt[name] += ret['erase_in_tokens'] * LINGUA_PRICE

            else:
                pass
                #print('WARN: noret', name, bug_id)

    return draw_table({
        'tot': dict(tot_cnt),

        'Keep%': {k: f"{v / tot_cnt[k] * 100 : .1f}" for k, v in keep_cnt.items()},

        'I': {k: f"{v / agent_in_cnt['baseline'] : .3f}" for k, v in agent_in_cnt.items()},
        'O': {k: f"{v / agent_in_cnt['baseline'] : .3f}" for k, v in agent_out_cnt.items()},
        '$': {k: f"{v / agent_cost_cnt['baseline'] : .3f}" for k, v in agent_cost_cnt.items()},
        '+$': {k: f"{v / agent_cost_cnt['baseline'] : .3f}" for k, v in overhead_cost_cnt.items()},

        'Pass%': {k: f"{v / tot_cnt[k] * 100 : .1f}" for k, v in pass_cnt.items()},
        'Step': {k: f"{v / tot_cnt[k] : .2f}" for k, v in step_cnt.items()},
        'PStep': {k: f"{v / pass_cnt[k] : .2f}" for k, v in passed_step_cnt.items()},

        #'+I': {k: round(v / agent_in_cnt['baseline'], 4) for k, v in overhead_in_cnt.items()},
        #'+O': {k: round(v / agent_in_cnt['baseline'], 4) for k, v in overhead_out_cnt.items()},
        'T$': {k: round((v + agent_cost_cnt[k]) / agent_cost_cnt['baseline'], 3) for k, v in overhead_cost_cnt.items()},
    })

In [9]:
export(subjs_appr_100, [
    ('claude35haiku', 'design_space/llm_claude35haiku', EXTRA_KEYS_OURS),
    ('gemini25flash', 'design_space/llm_gemini25flash', EXTRA_KEYS_OURS),
    ('gpt5mini', 'design_space/llm_gpt5mini', EXTRA_KEYS_OURS),
    ('deepseekv3', 'design_space/llm_deepseekv3', EXTRA_KEYS_OURS),
    ('qwen3', 'design_space/llm_qwen3', EXTRA_KEYS_OURS),

    ('baseline', 'design_space/baseline', []),
    ('lingua', 'design_space/play_lingua', EXTRA_KEYS_LINGUA),
    ('random', 'design_space/play_random', []),
    ('delete', 'design_space/play_delete', []),
])

100%|██████████| 100/100 [00:00<00:00, 173.78it/s]

     	claude35haiku  	gemini25flash  	gpt5mini  	deepseekv3  	qwen3  	baseline  	lingua  	random  	delete
tot  	100            	100            	100       	100         	100    	100       	100     	100     	100
Keep%	14.4           	21.9           	28.6      	23.7        	29.0   	          	22.5
I    	0.473          	0.553          	0.586     	0.606       	0.722  	1.000     	0.603   	0.642   	0.428
O    	0.012          	0.012          	0.012     	0.012       	0.012  	0.012     	0.013   	0.013   	0.013
$    	0.652          	0.698          	0.707     	0.733       	0.818  	1.000     	0.753   	0.794   	0.655
+$   	0.148          	0.078          	0.077     	0.052       	0.065  	          	0.000
Pass%	60.0           	63.0           	65.0      	64.0        	62.0   	65.0      	61.0    	64.0    	58.0
Step 	42.39          	40.95          	38.90     	40.41       	41.10  	39.74     	43.89   	44.95   	45.43
PStep	40.27          	39.46          	37.34     	38.47       	39.69  	38.29     	42.21   	43.6

In [10]:
export(subjs_appr_100, [
    ('baseline', 'design_space/baseline', []),
    ('≥0', 'design_space/threshold_0', EXTRA_KEYS_OURS),
    ('≥250', 'design_space/threshold_250', EXTRA_KEYS_OURS),
    ('≥500', 'design_space/llm_gpt5mini', EXTRA_KEYS_OURS),
    ('≥1000', 'design_space/threshold_1000', EXTRA_KEYS_OURS),
    ('≥2000', 'design_space/threshold_2000', EXTRA_KEYS_OURS),
])

100%|██████████| 100/100 [00:00<00:00, 244.30it/s]

     	baseline  	≥0   	≥250  	≥500  	≥1000  	≥2000
tot  	100       	100  	100   	100   	100    	100
Keep%	          	32.6 	32.0  	28.6  	24.3   	16.2
I    	1.000     	0.547	0.587 	0.586 	0.662  	0.728
O    	0.012     	0.011	0.011 	0.012 	0.012  	0.012
$    	1.000     	0.669	0.700 	0.707 	0.765  	0.816
+$   	          	0.118	0.100 	0.077 	0.040  	0.017
Pass%	65.0      	62.0 	65.0  	65.0  	60.0   	58.0
Step 	39.74     	38.57	39.68 	38.90 	39.70  	40.22
PStep	38.29     	37.40	38.09 	37.34 	38.27  	38.91
T$   	          	0.788	0.8   	0.784 	0.804  	0.833


In [10]:
export(subjs_appr_100, [
    ('baseline', 'design_space/baseline', []),
    ('0', 'design_space/turn_0', EXTRA_KEYS_OURS),
    ('1', 'design_space/turn_1', EXTRA_KEYS_OURS),
    ('2', 'design_space/llm_gpt5mini', EXTRA_KEYS_OURS),
    ('3', 'design_space/turn_3', EXTRA_KEYS_OURS),
])

100%|██████████| 100/100 [00:00<00:00, 294.74it/s]

     	baseline  	0    	1    	2    	3
tot  	100       	100  	100  	100  	100
Keep%	          	31.7 	31.0 	28.6 	31.3
I    	1.000     	0.727	0.569	0.586	0.624
O    	0.012     	0.013	0.011	0.012	0.012
$    	1.000     	0.843	0.688	0.707	0.736
+$   	          	0.081	0.067	0.077	0.078
Pass%	65.0      	59.0 	62.0 	65.0 	66.0
Step 	39.74     	44.94	39.13	38.90	40.25
PStep	38.29     	43.59	37.73	37.34	40.06
T$   	          	0.924	0.755	0.784	0.814


In [11]:
export(subjs_appr_100, [
    ('baseline', 'design_space/baseline', []),
    ('0', 'design_space/before_0', EXTRA_KEYS_OURS),
    ('1', 'design_space/llm_gpt5mini', EXTRA_KEYS_OURS),
    ('2', 'design_space/before_2', EXTRA_KEYS_OURS),
])

100%|██████████| 100/100 [00:00<00:00, 336.48it/s]

     	baseline  	0    	1    	2
tot  	100       	100  	100  	100
Keep%	          	31.5 	28.6 	34.5
I    	1.000     	0.644	0.586	0.719
O    	0.012     	0.012	0.012	0.011
$    	1.000     	0.749	0.707	0.790
+$   	          	0.075	0.077	0.078
Pass%	65.0      	64.0 	65.0 	65.0
Step 	39.74     	40.31	38.90	39.32
PStep	38.29     	39.45	37.34	37.49
T$   	          	0.825	0.784	0.868


In [12]:
export(subjs_multi_flash, [
    ('baseline', 'multi/baseline', []),
    ('gpt5mini', 'multi/llm_gpt5mini', EXTRA_KEYS_OURS),
])

100%|██████████| 300/300 [00:00<00:00, 462.80it/s]

     	baseline  	gpt5mini
tot  	300       	300
Keep%	          	30.2
I    	1.000     	0.596
O    	0.006     	0.006
$    	1.000     	0.676
+$   	          	0.055
Pass%	40.0      	39.0
Step 	70.37     	70.45
PStep	62.08     	60.77
T$   	          	0.731


In [13]:
export(subjs_eval_200, [
    ('baseline', 'eval/baseline_for_claude4', []),
    ('ours', 'eval/gpt5mini_for_claude4', EXTRA_KEYS_OURS),
])

100%|██████████| 200/200 [00:00<00:00, 702.19it/s]

     	baseline  	ours
tot  	200       	200
Keep%	          	30.8
I    	1.000     	0.601
O    	0.012     	0.011
$    	1.000     	0.714
+$   	          	0.074
Pass%	64.5      	66.5
Step 	39.62     	39.95
PStep	37.75     	38.21
T$   	          	0.789


In [14]:
export(subjs_eval_200, [
    ('baseline', 'eval/baseline_for_gemini25pro', []),
    ('ours', 'eval/gpt5mini_for_gemini25pro', EXTRA_KEYS_OURS),
], 'llm_gemini25pro')

100%|██████████| 200/200 [00:00<00:00, 626.31it/s]

     	baseline  	ours
tot  	200       	200
Keep%	          	22.6
I    	1.000     	0.591
O    	0.007     	0.006
$    	1.000     	0.623
+$   	          	0.118
Pass%	50.5      	52.0
Step 	37.98     	37.44
PStep	32.59     	31.72
T$   	          	0.741


In [15]:
export(subjs_multi_flash, [
    ('baseline', 'multi/baseline_for_gemini25pro', []),
    ('ours', 'multi/gpt5mini_for_gemini25pro', EXTRA_KEYS_OURS),
], 'llm_gemini25pro')

100%|██████████| 300/300 [00:00<00:00, 553.65it/s]

     	baseline  	ours
tot  	299       	299
Keep%	          	25.7
I    	1.000     	0.403
O    	0.007     	0.009
$    	1.000     	0.559
+$   	          	0.082
Pass%	21.7      	22.7
Step 	57.20     	43.90
PStep	37.86     	29.75
T$   	          	0.641


In [16]:
def export_cat(subjs, p_baseline, p_ours, fix_agent_name, analysis_agent_name):
    tot_cnt = Counter()
    dpass_cnt = Counter()
    dstep_cnt = Counter()
    passed_step_baseline_cnt = Counter()
    passed_step_ours_cnt = Counter()
    keep_cnt = Counter()
    agent_in_baseline_cnt = Counter()
    agent_in_ours_cnt = Counter()
    agent_out_baseline_cnt = Counter()
    agent_out_ours_cnt = Counter()
    agent_cost_baseline_cnt = Counter()
    agent_cost_ours_cnt = Counter()
    overhead_cost_cnt = Counter()

    AGENT_PRICE = ANALYSIS_PRICES[fix_agent_name]
    ANALYSIS_PRICE = ANALYSIS_PRICES[analysis_agent_name]

    for bug_id in tqdm(subjs):
        lang = subjs_details[bug_id]['language']
        ret_baseline = get_metrics(p_baseline, bug_id, [])
        ret_ours = get_metrics(p_ours, bug_id, EXTRA_KEYS_OURS)

        if not ret_baseline or not ret_ours:
            continue

        tot_cnt[lang] += 1

        pass_ours = 1 if ret_ours['result_val']=='pass' else 0
        pass_baseline = 1 if ret_baseline['result_val']=='pass' else 0
        dpass = pass_ours - pass_baseline
        dpass_cnt[lang] += dpass

        dstep_cnt[lang] += ret_ours['tot_step'] - ret_baseline['tot_step']
        if pass_ours:
            passed_step_ours_cnt[lang] += ret_ours['tot_step']
        if pass_baseline:
            passed_step_baseline_cnt[lang] += ret_baseline['tot_step']

        agent_in_baseline_cnt[lang] += ret_baseline['prompt_tokens']
        agent_in_ours_cnt[lang] += ret_ours['prompt_tokens']
        agent_out_baseline_cnt[lang] += ret_baseline['completion_tokens']
        agent_out_ours_cnt[lang] += ret_ours['completion_tokens']
        agent_cost_baseline_cnt[lang] += ret_baseline['prompt_tokens'] * AGENT_PRICE[0] + ret_baseline['prompt_tokens'] * .02 * AGENT_PRICE[1] + ret_baseline['completion_tokens'] * AGENT_PRICE[2]
        agent_cost_ours_cnt[lang] += ret_ours['prompt_tokens'] * AGENT_PRICE[0] + ret_ours['prompt_tokens'] * .02 * AGENT_PRICE[1] + ret_ours['completion_tokens'] * AGENT_PRICE[2]

        keep_cnt[lang] += (ret_ours['erase_out_tokens'] / ret_ours['erase_in_tokens']) if ret_ours['erase_in_tokens'] else 1

        adjustment = ret_ours['analysis_count'] * ANALYSIS_SYS_PROMPT_TOKENS
        overhead_cost_cnt[lang] += ret_ours['analysis_prompt_tokens'] * ANALYSIS_PRICE[1] - adjustment * (ANALYSIS_PRICE[1] - ANALYSIS_PRICE[0]) + ret_ours['analysis_completion_tokens'] * ANALYSIS_PRICE[2]

    #print(keep_cnt)
    return draw_table({
        '#Inst': dict(tot_cnt),

        'Keep%': {k: f"{v / tot_cnt[k] * 100 : .1f}" for k, v in keep_cnt.items()},

        'I': {k: f"{v / agent_in_baseline_cnt[k] : .3f}" for k, v in agent_in_ours_cnt.items()},
        'O': {k: f"{v / agent_in_baseline_cnt[k] : .3f}" for k, v in agent_out_ours_cnt.items()},
        '$': {k: f"{v / agent_cost_baseline_cnt[k] : .3f}" for k, v in agent_cost_ours_cnt.items()},
        '+$': {k: f"{v / agent_cost_baseline_cnt[k] : .3f}" for k, v in overhead_cost_cnt.items()},

        'dPass': dict(dpass_cnt),
        'dStep': {k: f"{(v / tot_cnt[k]) : .2f}" for k, v in dstep_cnt.items()},
        'dPStep': {k: f"{(v - passed_step_baseline_cnt[k]) / tot_cnt[k] : .2f}" for k, v in passed_step_ours_cnt.items()},
    })

In [17]:
export_cat(subjs_multi_flash, 'multi/baseline', 'multi/llm_gpt5mini', 'llm_claude4sonnet', 'llm_gpt5mini')

export_cat(subjs_multi_flash, 'multi/baseline_for_gemini25pro', 'multi/gpt5mini_for_gemini25pro', 'llm_gemini25pro', 'llm_gpt5mini')

100%|██████████| 300/300 [00:00<00:00, 451.67it/s]


      	c    	rust  	typescript  	java  	go   	javascript  	c++
#Inst 	40   	45    	45          	40    	45   	45          	40
Keep% 	24.7 	28.1  	29.9        	31.7  	34.0 	29.8        	32.7
I     	0.543	0.603 	0.560       	0.637 	0.615	0.649       	0.602
O     	0.005	0.006 	0.006       	0.007 	0.006	0.010       	0.006
$     	0.620	0.680 	0.635       	0.723 	0.646	0.806       	0.679
+$    	0.043	0.061 	0.049       	0.057 	0.049	0.085       	0.049
dPass 	-1   	-2    	1           	1     	-3   	1           	0
dStep 	0.40 	-0.27 	-1.11       	-1.32 	0.13 	3.11        	-0.53
dPStep	-2.00	-3.67 	0.67        	2.30  	-4.84	1.24        	-1.38


100%|██████████| 300/300 [00:00<00:00, 519.04it/s]

      	c     	rust  	typescript  	java  	go   	javascript  	c++
#Inst 	39    	44    	45          	40    	45   	45          	40
Keep% 	18.3  	22.3  	25.5        	26.1  	28.1 	34.1        	24.4
I     	0.443 	0.310 	0.525       	0.330 	0.656	0.459       	0.298
O     	0.009 	0.007 	0.015       	0.008 	0.013	0.010       	0.006
$     	0.638 	0.425 	0.746       	0.438 	0.885	0.597       	0.424
+$    	0.097 	0.072 	0.105       	0.081 	0.114	0.083       	0.055
dPass 	-3    	0     	3           	-1    	1    	4           	-1
dStep 	-10.15	-13.91	-11.36      	-15.88	-3.02	-11.44      	-27.52
dPStep	-1.67 	-1.64 	0.58        	-5.10 	-0.76	1.02        	-3.38
